In [ ]:
# __ri_prediction_notebook_setup__
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
TRAINING_DATA_DIR = DATA_DIR / "training"
CASE_DATA_DIR = DATA_DIR / "cases"
EXAMPLE_DATA_DIR = DATA_DIR / "examples"
MODEL_DIR = PROJECT_ROOT / "models" / "pretrained"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
import pandas as pd
import math
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn import preprocessing, svm, neighbors
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
#
# reading all input data, and replace/remove bad data. Note that this data set 
# must contain the last column "class" that indicates RI (1) or non-RI (0).
#
flag_input_more_time="yes"
if flag_input_more_time == "yes":
    df = pd.read_csv(TRAINING_DATA_DIR / "SHIP_AllBasinsMaster_12h.csv")
    df.drop(['OHC','OHC(+6h)','OHC(+12h)'], axis=1, inplace=True)
else:
    df = pd.read_csv(TRAINING_DATA_DIR / "SHIP_AtlanticMajorHurricane2016-2019.csv")
    df.drop(['OHC'], axis=1, inplace=True)
#df.replace(9999,-1111, inplace=True)
#df.replace('?',-99999, inplace=True)    
print(df.head(10))
x = np.array(df.drop(['class'],axis=1))
y = np.array(df['class'])
print(x[0])
print(y[0])
print("data length is" ,len(df))

In [ ]:
#
# splitting total dataframe into RI and non-RI data for later dealing with unbalanced
# data set.
#
df_ri = []
df_nonri = []
for i in range(len(df)):
    if df['class'][i] == 1:
        df_ri.append(df.loc[i,:])
    else:
        df_nonri.append(df.loc[i,:])  
print(len(df_ri),len(df_nonri), df_ri[50]['SST'],df_nonri[50]['SST(+12h)'])        

In [ ]:
#
# create a function that randomly selects from non-RI data (df_nonri) a net set 
# with the same length as RI data (df_ri). The training and prediction will be carried out only 
# for the combined set (update_x_data) to maintain the balanced data between RI and non-RI.
# No need to create y data, as the df data contains the label in the last column.
#
import random
def generate_data(df_ri,df_nonri):    
    my_list = list(range(1,len(df_nonri)))                               
    random.shuffle(my_list)
    new_x_data = []
    new_y_data = []
    #
    # randomize the nonRI data first
    #
    for i in my_list[:len(df_ri)]:
        new_x_data.append(df_nonri[i])
        #new_y_data.append(df_nonri[i]['class'])
    #
    # join nonRI and RI data into a single list
    #
    for j in df_ri:
        new_x_data.append(j)    
        #new_y_data.append(t['class'])        
    #
    # randomize the final list
    #
    new_list = list(range(len(new_x_data)))
    random.shuffle(new_list)
    #print(new_list)
    update_x_data = []
    #update_y_data = []
    for i in new_list:
        update_x_data.append(new_x_data[i])        
        #update_y_data.append(new_y_data[i])        
    return update_x_data

In [ ]:
#
# reading a new SHIP record to make a prediction
#
if flag_input_more_time == "yes":
    df = pd.read_csv(EXAMPLE_DATA_DIR / "ship_input_12h.csv")
    df.drop(['OHC','OHC(+6h)','OHC(+12h)'], axis=1, inplace=True)
else:
    df = pd.read_csv(EXAMPLE_DATA_DIR / "ship_input_00h.csv")
    df.drop(['OHC'], axis=1, inplace=True)
df.replace('?',-99999, inplace=True)
x_fcst = np.array(df.drop(['class'],axis=1))
y_fcst = np.array(df['class'])
print('Test data length is: ',len(x_fcst))

In [ ]:
#
# training for each realization and make a forecast for each realization. 
#
num_realizations = 20
forecast = np.zeros((len(x_fcst),num_realizations))
for realization in range(num_realizations):
    z = np.array(generate_data(df_ri,df_nonri))    
    n_sample = len(z)
    n_predictors = len(z[0])-1
    x_new = np.zeros((n_sample,n_predictors))
    y_new = np.zeros(n_sample)
    x_new[:,:n_predictors] = z[:,:n_predictors]
    y_new[:] = z[:,n_predictors]
    #print(len(z),len(z[0]),z[0,15],z[50,15],z[54,15],z[101,15])
    #print(y_new[0],y_new[50],y_new[54],y_new[101])
    #print(x_new[10],y_new[10])
    
    x_train, x_test, y_train, y_test = train_test_split(x_new, y_new, test_size=0.1)
    clf = neighbors.KNeighborsClassifier()
    clf.fit(x_train, y_train)
    accuracy = clf.score(x_test, y_test)
    pred = clf.predict(x_test)
    print("The accuracy for realization ",realization," is", accuracy)
    #print(classification_report(y_test,pred,zero_division=1))    
    cm = pd.DataFrame(confusion_matrix(y_test,pred))
    
    single_fcst = clf.predict(x_fcst)
    forecast[:,realization] = single_fcst
    

In [ ]:
for i in range(len(forecast)):
    print("Actual RI",y_fcst[i]," <---> ",forecast[i,:]," Predicted RI Propbability: ",sum(forecast[i,:])/num_realizations)
